## Table of Contents

- [Problem Statement](#problem-statement)
  - [Business Context](#business-context)
  - [Objective](#objective)
  - [Data Description](#data-description)
- [Libraries and Dependencies](#libraries-and-dependencies)
- [SETUP](#setup)
- [Model Setup, Optimization and Test](#model-setup-optimization-and-test)
  - [The model setup](#the-model-setup)
  - [Model Response](#model-response)
  - [Test Model for optimal values](#test-model-for-optimal-values)
    - [Max Tokens Optimization](#max-tokens-optimization)
      - [Max Token - Conclusion](#max-token---conclusion)
    - [top_k Optimization:](#topk-optimization)
      - [top_k conclusion](#topk-conclusion)
    - [Temperature Optimization](#temperature-optimization)
      - [Temperature Conclusion](#temperature-conclusion)
    - [top_p Optimization](#topp-optimization)
      - [top_p conclusion](#topp-conclusion)
    - [Test all queries with optimal values](#test-all-queries-with-optimal-values)
- [LLM with Prompt Engineering](#llm-with-prompt-engineering)
- [Data Preparation for RAG](#data-preparation-for-rag)
  - [Data Load](#data-load)
  - [Data Overview](#data-overview)
    - [Checking the first 5 pages](#checking-the-first-5-pages)
    - [Checking the number of pages](#checking-the-number-of-pages)
  - [Data Chunking](#data-chunking)
  - [Data Embedding](#data-embedding)
  - [Vector Database](#vector-database)
  - [Similarity Search Check](#similarity-search-check)
  - [Retriever Check](#retriever-check)
  - [LLM Response Check](#llm-response-check)
  - [RAG Response Function](#rag-response-function)
- [RAG without fine tuning](#rag-without-fine-tuning)
- [RAG with fine-tuning](#rag-with-fine-tuning)
- [LLM-as-a-judge - Output Evaluation](#llm-as-a-judge---output-evaluation)
  - [Grounding Function](#grounding-function)
  - [LLM-as-a-judge](#llm-as-a-judge)
- [All Outputs](#all-outputs)
- [Actionable Insights and Business Recommendations](#actionable-insights-and-business-recommendations)
  - [Overview of Tuning Combinations](#overview-of-tuning-combinations)
  - [Key Takeaways for the Business](#key-takeaways-for-the-business)
- [Export](#export)

## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Libraries and Dependencies

In [1]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
#!CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python --force-reinstall --no-cache-dir -q

#Use this command command for Mac (Apple Silicon)
#!CMAKE_ARGS="-DGGML_METAL=on" FORCE_CMAKE=1 pip install llama-cpp-python # --no-cache-dir --force-reinstall # Uncomment if you want a fresh resinstall.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 249.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 269.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 373.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 335.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 283.3 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for llama-cpp-python (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for llama-cpp-python
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (llama-cpp-python)


In [2]:
# For installing the libraries & downloading models from HF Hub
!pip install huggingface_hub pandas llama-index llama-cpp-python tiktoken pymupdf langchain langchain-community chromadb sentence-transformers numpy -q

# These are already installed in the workspace, so no need to install again. 

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 28.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import json,os
import pandas as pd
from IPython.core.display import display
from IPython.core.display import HTML

import tiktoken

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

## SETUP

In [4]:
# 7B model from Mistral Instruct
MODEL_PATH = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
MODEL_BASENAME = "mistral-7b-instruct-v0.2.Q6_K.gguf"

# https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2
# This is a sentence-transformers model: 
# It maps sentences & paragraphs to a 384 dimensional dense vector space and can be used for tasks like clustering or semantic search.

EMBEDDING_MODEL = "all-MiniLM-L6-v2"
VECTOR_DB = 'medical_db'

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100
CHUNK_ENCODING = 'cl100k_base'


# Create Vector DB location if does not exist
if not os.path.exists(VECTOR_DB):
  os.makedirs(VECTOR_DB)

PDF_PATH = '/kaggle/input/datasets/assignarc/medical-dataset/medical_diagnosis_manual.pdf'
#PDF_PATH = "medical_diagnosis_manual.pdf" 

# Capture responses for later comparision
responses =pd.DataFrame(columns=["Type","Run","Query","Response", "Grounding","Relevance"])

#List of queries for tesing
queries  = [
    "What is the protocol for managing sepsis in a critical care unit?",
    "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",
    "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",
    "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",
    "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?" ,
]

# System Prompt for LLM Queries
SYSTEM_PROMPT = """
    You are a helpful and knowledgeable medical assistant. 
    Answer the user's questions clearly, concisely, and based on the provided medical information.
    """

#System and User prompt templates
QNA_SYSTEM_PROMPT = "You are an expert medical assistant. Provide accurate and concise medical advice only based on the context provided." 
QNA_USER_MESSAGE_TEMPLATE = "Context: {context}\n\nQuestion: {question}\n\n" 


#LLM as judge parameters
GROUNDNESS_RATE_SYSTEM_MESSAGE = """ 
    You are a professional medical evaluator and tasked with rating AI generated answers to questions posed by users. 
    Please rate the groundedness of the model's answer based on the provided context.
    
    Evaluation criteria:
    The task is to judge the extent to which the metric is followed by the answer.
        1 - The metric is not followed at all
        2 - The metric is followed only to a limited extent
        3 - The metric is followed to a good extent
        4 - The metric is followed mostly
        5 - The metric is followed completely

    Metric:
    The answer should be accurate and concise and should be derived only from the information presented in the context

    Instructions:
    1. First write down the steps that are needed to evaluate the answer as per the metric.
    2. Give a step-by-step explanation if the answer adheres to the metric considering the question and context as the input.
    3. Next, evaluate the extent to which the metric is followed.
    4. Use the previous information to rate the answer using the evaluaton criteria and assign a score.
"""


RELEVENCE_RATER_SYSTEM_MESSAGE = """
        You are a professional medical evaluator. Rate the relevance of the answer to the user's question.
        You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
        In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

        Evaluation criteria:
        The task is to judge the extent to which the metric is followed by the answer.
        1 - The metric is not followed at all
        2 - The metric is followed only to a limited extent
        3 - The metric is followed to a good extent
        4 - The metric is followed mostly
        5 - The metric is followed completely

        Metric:
        Relevance measures how well the answer addresses the main aspects of the question, based on the context in accurate and concise manner.
        Consider whether all and only the important aspects are contained in the answer when evaluating relevance.

        Instructions:
        1. First write down the steps that are needed to evaluate the context as per the metric.
        2. Give a step-by-step explanation if the context adheres to the metric considering the question as the input.
        3. Next, evaluate the extent to which the metric is followed.
        4. Use the previous information to rate the context using the evaluaton criteria and assign a score.

    """ 



GROUNDNESS_USER_MESSAGE_TEMPLATE = """
    ###Question
    {question}

    ###Context
    {context}

    ###Answer
    {answer}
"""


## Model Setup, Optimization and Test

### The model setup

In [5]:
model_path = hf_hub_download(
    repo_id= MODEL_PATH, 
    filename= MODEL_BASENAME
)

llm = Llama(
    model_path=model_path,
    n_ctx=8192,
    n_gpu_layers=38,
    n_batch=512
)

#uncomment the below snippet of code if the runtime is connected to CPU only.
#llm = Llama(
#    model_path=model_path,
#    n_ctx=8192,
#    n_cores=-2
#)

mistral-7b-instruct-v0.2.Q6_K.gguf:   0%|          | 0.00/5.94G [00:00<?, ?B/s]

llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from /root/.cache/huggingface/hub/models--TheBloke--Mistral-7B-Instruct-v0.2-GGUF/snapshots/3a6fbf4a41a1d52e415a4958cde6856d34b2db93/mistral-7b-instruct-v0.2.Q6_K.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.2
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loade

### Model Response

In [6]:
def LLM_response(
    query:str,
    max_tokens:int=1024,
    temperature:float=0.0,
    top_p:float=0.95,
    top_k:int=50,
    print :bool = False):
    
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )
    if(print):
      display(HTML(
            f"<h3>{query}</h3>" +
            f"<h4>{llm.metadata['general.name']} | Tokens : {max_tokens} | Temp : {temperature} | top_p : {top_p}, | top_k : {top_k} </h4> " +
            f"<pre style=\"white-space:pre-line;\">{model_output['choices'][0]['text'].replace("\n","<p>")}</pre>"
          )
       )
    return model_output['choices'][0]['text']

### Test Model for optimal values

#### Max Tokens Optimization

In [7]:
max_tokens_testset = {128,256,512,1024}
for index, tkn in enumerate(max_tokens_testset):
    LLM_response(query="What treatment options are available for managing hypertension?", max_tokens=tkn, print=True)

llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =    4455.19 ms /    12 tokens (  371.27 ms per token,     2.69 tokens per second)
llama_perf_context_print:        eval time =   51219.57 ms /   127 runs   (  403.30 ms per token,     2.48 tokens per second)
llama_perf_context_print:       total time =   55743.85 ms /   139 tokens
llama_perf_context_print:    graphs reused =        122


Llama.generate: 11 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =  102064.23 ms /   256 runs   (  398.69 ms per token,     2.51 tokens per second)
llama_perf_context_print:       total time =  102231.92 ms /   257 tokens
llama_perf_context_print:    graphs reused =        247


Llama.generate: 11 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =  205415.14 ms /   512 runs   (  401.20 ms per token,     2.49 tokens per second)
llama_perf_context_print:       total time =  205877.89 ms /   513 tokens
llama_perf_context_print:    graphs reused =        495


Llama.generate: 11 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =  210341.34 ms /   526 runs   (  399.89 ms per token,     2.50 tokens per second)
llama_perf_context_print:       total time =  210818.60 ms /   527 tokens
llama_perf_context_print:    graphs reused =        509


##### Max Token - Conclusion
* Looking at outputs 512 seems like a very decent degree of output. Remains same at 1024
* Anything less than 512 is getting chopped off. but in many cases 512 may not be sufficient as well, so seting to 1024.

In [8]:
# Set Optimal value for Max Tokens
OPTIMAL_MAX_TOKENS = 1024

#### top_k Optimization: 
* When an LLM generates text, it calculates probabilities for all possible next words (tokens). If top_k is set, the model ranks these, keeps only the top, and redistributes the probability mass among them.
* Low top_k (e.g., 1–10): Leads to more focused, coherent, and deterministic, but sometimes repetitive, responses. A top_k of 1 is equivalent to "greedy decoding," where the model always chooses the single most likely next word.
* High top_k (e.g., 50–100): Increases diversity and creativity because the model has more options to choose from, but it risks lower coherence or less relevant outputs.
Try: Often set around 40-50, allowing for a balance between coherence and variety.

In [9]:
top_k_testset = {10, 25,50,75,100}
for index, k in enumerate(top_k_testset):
    LLM_response(query="What treatment options are available for managing hypertension?", max_tokens=OPTIMAL_MAX_TOKENS, top_k=k, print=True)

Llama.generate: 11 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =  211010.99 ms /   526 runs   (  401.16 ms per token,     2.49 tokens per second)
llama_perf_context_print:       total time =  211488.56 ms /   527 tokens
llama_perf_context_print:    graphs reused =        509


Llama.generate: 11 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =  210689.36 ms /   526 runs   (  400.55 ms per token,     2.50 tokens per second)
llama_perf_context_print:       total time =  211169.38 ms /   527 tokens
llama_perf_context_print:    graphs reused =        509


Llama.generate: 11 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =  211782.74 ms /   526 runs   (  402.63 ms per token,     2.48 tokens per second)
llama_perf_context_print:       total time =  212266.44 ms /   527 tokens
llama_perf_context_print:    graphs reused =        509


Llama.generate: 11 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =  211764.43 ms /   526 runs   (  402.59 ms per token,     2.48 tokens per second)
llama_perf_context_print:       total time =  212244.39 ms /   527 tokens
llama_perf_context_print:    graphs reused =        509


Llama.generate: 11 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =  210482.96 ms /   526 runs   (  400.16 ms per token,     2.50 tokens per second)
llama_perf_context_print:       total time =  210961.23 ms /   527 tokens
llama_perf_context_print:    graphs reused =        509


##### top_k conclusion 
* Not much of difference observed in this. So leaving it setting to 10

In [10]:
# Optimal top_k 
OPTIMAL_TOP_K = 10

#### Temperature Optimization
Temperature in AI model performance acts as a "creativity dial" that controls the randomness and predictability of outputs, typically ranging from 0 to 1 or higher. A low temperature makes the model deterministic, safe, and focused on high-probability, accurate responses. A high temperature encourages diverse, creative, and varied outputs by increasing the probability of picking less likely words. 

* Low Temperature (0.0-0.3): Ideal for tasks requiring precision, such as coding, factual Q&A, or data extraction. The model behaves "greedily," choosing the most probable token consistently.
* Medium Temperature (0.5-0.8): Suitable for balanced tasks like blog writing or chatbots that need a mix of coherence and creativity.
* High Temperature (0.8-1.0+): Best for creative writing, brainstorming, or artistic tasks where unexpected, novel, or random output is desired.
Impact on Accuracy: Higher temperatures can reduce accuracy by causing the model to take more risks, leading to potential hallucinations or nonsensical outputs, with studies showing 1.0 temperature can be 28-73% less accurate than lower settings.

In [11]:
temperature_testset = [0.1,0.2,0.6,0.7,0.8,1.0]
for index, t in enumerate(temperature_testset):
    LLM_response(query="What treatment options are available for managing hypertension?", 
                max_tokens=OPTIMAL_MAX_TOKENS, 
                temperature=t,
                top_k=OPTIMAL_TOP_K, 
                print=True)

Llama.generate: 11 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =  215476.75 ms /   531 runs   (  405.79 ms per token,     2.46 tokens per second)
llama_perf_context_print:       total time =  215964.57 ms /   532 tokens
llama_perf_context_print:    graphs reused =        514


Llama.generate: 11 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =  249230.69 ms /   612 runs   (  407.24 ms per token,     2.46 tokens per second)
llama_perf_context_print:       total time =  249831.05 ms /   613 tokens
llama_perf_context_print:    graphs reused =        592


Llama.generate: 11 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =  353774.11 ms /   861 runs   (  410.89 ms per token,     2.43 tokens per second)
llama_perf_context_print:       total time =  354808.31 ms /   862 tokens
llama_perf_context_print:    graphs reused =        833


Llama.generate: 11 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =  162878.54 ms /   403 runs   (  404.17 ms per token,     2.47 tokens per second)
llama_perf_context_print:       total time =  163197.38 ms /   404 tokens
llama_perf_context_print:    graphs reused =        390


Llama.generate: 11 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =  200026.98 ms /   496 runs   (  403.28 ms per token,     2.48 tokens per second)
llama_perf_context_print:       total time =  200454.93 ms /   497 tokens
llama_perf_context_print:    graphs reused =        480


Llama.generate: 11 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =  290632.86 ms /   719 runs   (  404.22 ms per token,     2.47 tokens per second)
llama_perf_context_print:       total time =  291411.38 ms /   720 tokens
llama_perf_context_print:    graphs reused =        696


##### Temperature Conclusion 
* For this exercise 0.8 provides best results, answering question to the best. 

In [12]:
OPTIMAL_TEMPERATURE = 0.8

#### top_p Optimization
* Nucleus Sampling: Instead of a fixed number of tokens (like Top-K), top_p dynamically adjusts the candidate pool size based on the model's confidence in the next word.
* Controlling Randomness: A top_p value of 0.1 means only the tokens comprising the top 10% probability mass are considered, resulting in safer, more concise, and repetitive output. A higher value (e.g., 0.9 or 1.0) allows for a wider range of tokens, leading to more creative, diverse, or unpredictable content.
Performance Trade-offs:
* Low top_p (e.g., < 0.5): Increases coherence and consistency, which is generally better for factual, coding, or structured tasks.
* High top_p (e.g., > 0.8): Encourages creativity but may lead to lower coherence, hallucinations, or "off-the-rails" text.
* Interaction with Temperature: Both top_p and Temperature affect output randomness. Generally, it is advised to adjust one or the other, rather than both simultaneously, to maintain control over output quality.

In [13]:
top_p_testset = [0.2,0.4,0.5,0.8,1.0]
for index, p in enumerate(top_p_testset):
    LLM_response(query="What treatment options are available for managing hypertension?", 
                max_tokens=OPTIMAL_MAX_TOKENS, 
                temperature=OPTIMAL_TEMPERATURE,
                top_k=OPTIMAL_TOP_K, 
                top_p=p,
                print=True)

Llama.generate: 11 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =  213493.67 ms /   526 runs   (  405.88 ms per token,     2.46 tokens per second)
llama_perf_context_print:       total time =  213962.78 ms /   527 tokens
llama_perf_context_print:    graphs reused =        509


Llama.generate: 11 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =  315286.77 ms /   786 runs   (  401.13 ms per token,     2.49 tokens per second)
llama_perf_context_print:       total time =  316163.33 ms /   787 tokens
llama_perf_context_print:    graphs reused =        761


Llama.generate: 11 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =  241065.29 ms /   600 runs   (  401.78 ms per token,     2.49 tokens per second)
llama_perf_context_print:       total time =  241637.10 ms /   601 tokens
llama_perf_context_print:    graphs reused =        580


Llama.generate: 11 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =  278977.96 ms /   701 runs   (  397.97 ms per token,     2.51 tokens per second)
llama_perf_context_print:       total time =  279700.18 ms /   702 tokens
llama_perf_context_print:    graphs reused =        678


Llama.generate: 11 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =  205726.71 ms /   512 runs   (  401.81 ms per token,     2.49 tokens per second)
llama_perf_context_print:       total time =  206179.93 ms /   513 tokens
llama_perf_context_print:    graphs reused =        495


##### top_p conclusion
* 0.2 seen to provide best possible results. 

In [14]:
#Optimal top_p value
OPTIMAL_TOP_P = 0.2

#### Test all queries with optimal values

In [15]:
for i, query in enumerate(queries):
    responses.loc[len(responses)] = [
        "1.TEST",
        i+1,
        query,
        LLM_response(query=query, max_tokens= OPTIMAL_MAX_TOKENS, temperature=OPTIMAL_TEMPERATURE, top_k=OPTIMAL_TOP_K, top_p=OPTIMAL_TOP_P, print=True),
        "",
        ""
    ]
# for index, response in responses.iterrows():
#     display(HTML(
#         f"<h3>{response['Type']} - Query {response['Run']}</h3>" + 
#         f"<blockquote>{response['Query']}</blockquote>" +
#         f"<blockquote>{response['Response']}</blockquote>"
#     ))
  

Llama.generate: 2 prefix-match hit, remaining 14 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =    2717.66 ms /    14 tokens (  194.12 ms per token,     5.15 tokens per second)
llama_perf_context_print:        eval time =  297967.27 ms /   731 runs   (  407.62 ms per token,     2.45 tokens per second)
llama_perf_context_print:       total time =  301484.10 ms /   745 tokens
llama_perf_context_print:    graphs reused =        707


Llama.generate: 2 prefix-match hit, remaining 32 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =    6219.26 ms /    32 tokens (  194.35 ms per token,     5.15 tokens per second)
llama_perf_context_print:        eval time =  187895.78 ms /   467 runs   (  402.35 ms per token,     2.49 tokens per second)
llama_perf_context_print:       total time =  194505.16 ms /   499 tokens
llama_perf_context_print:    graphs reused =        452


Llama.generate: 4 prefix-match hit, remaining 34 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =    6615.38 ms /    34 tokens (  194.57 ms per token,     5.14 tokens per second)
llama_perf_context_print:        eval time =  290410.74 ms /   717 runs   (  405.04 ms per token,     2.47 tokens per second)
llama_perf_context_print:       total time =  297792.69 ms /   751 tokens
llama_perf_context_print:    graphs reused =        694


Llama.generate: 2 prefix-match hit, remaining 28 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =    5350.95 ms /    28 tokens (  191.11 ms per token,     5.23 tokens per second)
llama_perf_context_print:        eval time =  207633.99 ms /   515 runs   (  403.17 ms per token,     2.48 tokens per second)
llama_perf_context_print:       total time =  213447.23 ms /   543 tokens
llama_perf_context_print:    graphs reused =        497


Llama.generate: 2 prefix-match hit, remaining 35 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =    6798.91 ms /    35 tokens (  194.25 ms per token,     5.15 tokens per second)
llama_perf_context_print:        eval time =  220411.25 ms /   543 runs   (  405.91 ms per token,     2.46 tokens per second)
llama_perf_context_print:       total time =  227713.03 ms /   578 tokens
llama_perf_context_print:    graphs reused =        525


## LLM with Prompt Engineering

In [16]:
for i, query in enumerate(queries):
    responses.loc[len(responses)] = [
        "2.PROMPT_ENGINEERING",
        i+1,
        SYSTEM_PROMPT + "\n" + query,
        LLM_response(query=SYSTEM_PROMPT + "\n" + query, max_tokens= OPTIMAL_MAX_TOKENS, temperature=OPTIMAL_TEMPERATURE, top_k=OPTIMAL_TOP_K, top_p=OPTIMAL_TOP_P, print=True),
        "",
        ""
    ]

Llama.generate: 1 prefix-match hit, remaining 53 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =   10199.36 ms /    53 tokens (  192.44 ms per token,     5.20 tokens per second)
llama_perf_context_print:        eval time =  266431.64 ms /   654 runs   (  407.39 ms per token,     2.45 tokens per second)
llama_perf_context_print:       total time =  277308.27 ms /   707 tokens
llama_perf_context_print:    graphs reused =        632


Llama.generate: 40 prefix-match hit, remaining 32 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =    6142.38 ms /    32 tokens (  191.95 ms per token,     5.21 tokens per second)
llama_perf_context_print:        eval time =  114415.64 ms /   288 runs   (  397.28 ms per token,     2.52 tokens per second)
llama_perf_context_print:       total time =  120748.72 ms /   320 tokens
llama_perf_context_print:    graphs reused =        278


Llama.generate: 42 prefix-match hit, remaining 34 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =    6463.28 ms /    34 tokens (  190.10 ms per token,     5.26 tokens per second)
llama_perf_context_print:        eval time =  197952.61 ms /   494 runs   (  400.71 ms per token,     2.50 tokens per second)
llama_perf_context_print:       total time =  204843.23 ms /   528 tokens
llama_perf_context_print:    graphs reused =        478


Llama.generate: 40 prefix-match hit, remaining 28 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =    5518.01 ms /    28 tokens (  197.07 ms per token,     5.07 tokens per second)
llama_perf_context_print:        eval time =  139266.99 ms /   345 runs   (  403.67 ms per token,     2.48 tokens per second)
llama_perf_context_print:       total time =  145038.45 ms /   373 tokens
llama_perf_context_print:    graphs reused =        334


Llama.generate: 40 prefix-match hit, remaining 35 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =    6770.07 ms /    35 tokens (  193.43 ms per token,     5.17 tokens per second)
llama_perf_context_print:        eval time =  182646.47 ms /   452 runs   (  404.09 ms per token,     2.47 tokens per second)
llama_perf_context_print:       total time =  189796.39 ms /   487 tokens
llama_perf_context_print:    graphs reused =        437


In [17]:
for i, query in enumerate(queries):
    responses.loc[len(responses)] = [
        "2.PROMPT_ENG_T_0.7",
        i+1,
        SYSTEM_PROMPT + "\n" + query,
        LLM_response(query=SYSTEM_PROMPT + "\n" + query,max_tokens= OPTIMAL_MAX_TOKENS, temperature=0.7, top_k=OPTIMAL_TOP_K, top_p=OPTIMAL_TOP_P, print=True),
        "",
        ""
    ]

Llama.generate: 40 prefix-match hit, remaining 14 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =    2747.48 ms /    14 tokens (  196.25 ms per token,     5.10 tokens per second)
llama_perf_context_print:        eval time =  266400.42 ms /   654 runs   (  407.34 ms per token,     2.45 tokens per second)
llama_perf_context_print:       total time =  269819.70 ms /   668 tokens
llama_perf_context_print:    graphs reused =        632


Llama.generate: 40 prefix-match hit, remaining 32 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =    6304.31 ms /    32 tokens (  197.01 ms per token,     5.08 tokens per second)
llama_perf_context_print:        eval time =  116085.46 ms /   288 runs   (  403.07 ms per token,     2.48 tokens per second)
llama_perf_context_print:       total time =  122582.93 ms /   320 tokens
llama_perf_context_print:    graphs reused =        278


Llama.generate: 42 prefix-match hit, remaining 34 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =    6672.18 ms /    34 tokens (  196.24 ms per token,     5.10 tokens per second)
llama_perf_context_print:        eval time =  200387.10 ms /   494 runs   (  405.64 ms per token,     2.47 tokens per second)
llama_perf_context_print:       total time =  207488.33 ms /   528 tokens
llama_perf_context_print:    graphs reused =        478


Llama.generate: 40 prefix-match hit, remaining 28 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =    5391.04 ms /    28 tokens (  192.54 ms per token,     5.19 tokens per second)
llama_perf_context_print:        eval time =  139514.00 ms /   345 runs   (  404.39 ms per token,     2.47 tokens per second)
llama_perf_context_print:       total time =  145161.86 ms /   373 tokens
llama_perf_context_print:    graphs reused =        334


Llama.generate: 40 prefix-match hit, remaining 35 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =    6811.10 ms /    35 tokens (  194.60 ms per token,     5.14 tokens per second)
llama_perf_context_print:        eval time =  183431.55 ms /   452 runs   (  405.82 ms per token,     2.46 tokens per second)
llama_perf_context_print:       total time =  190621.68 ms /   487 tokens
llama_perf_context_print:    graphs reused =        437


In [18]:
for i, query in enumerate(queries):
    responses.loc[len(responses)] = [
        "2.PROMPT_ENG_P_0.8",
        i+1,
        SYSTEM_PROMPT + "\n" + query,
        LLM_response(query=SYSTEM_PROMPT + "\n" + query, max_tokens= OPTIMAL_MAX_TOKENS, temperature=OPTIMAL_TEMPERATURE, top_k=OPTIMAL_TOP_K, top_p=0.8, print=True),
        "",
        ""
    ]

Llama.generate: 40 prefix-match hit, remaining 14 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =    2645.12 ms /    14 tokens (  188.94 ms per token,     5.29 tokens per second)
llama_perf_context_print:        eval time =  168783.33 ms /   419 runs   (  402.82 ms per token,     2.48 tokens per second)
llama_perf_context_print:       total time =  171766.07 ms /   433 tokens
llama_perf_context_print:    graphs reused =        405


Llama.generate: 40 prefix-match hit, remaining 32 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =    6281.20 ms /    32 tokens (  196.29 ms per token,     5.09 tokens per second)
llama_perf_context_print:        eval time =   88627.43 ms /   219 runs   (  404.69 ms per token,     2.47 tokens per second)
llama_perf_context_print:       total time =   95043.54 ms /   251 tokens
llama_perf_context_print:    graphs reused =        211


Llama.generate: 42 prefix-match hit, remaining 34 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =    6642.89 ms /    34 tokens (  195.38 ms per token,     5.12 tokens per second)
llama_perf_context_print:        eval time =  236714.40 ms /   579 runs   (  408.83 ms per token,     2.45 tokens per second)
llama_perf_context_print:       total time =  243909.31 ms /   613 tokens
llama_perf_context_print:    graphs reused =        560


Llama.generate: 40 prefix-match hit, remaining 28 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =    5333.40 ms /    28 tokens (  190.48 ms per token,     5.25 tokens per second)
llama_perf_context_print:        eval time =  159855.18 ms /   398 runs   (  401.65 ms per token,     2.49 tokens per second)
llama_perf_context_print:       total time =  165499.01 ms /   426 tokens
llama_perf_context_print:    graphs reused =        385


Llama.generate: 40 prefix-match hit, remaining 35 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =    7426.32 ms /    35 tokens (  212.18 ms per token,     4.71 tokens per second)
llama_perf_context_print:        eval time =  165803.60 ms /   406 runs   (  408.38 ms per token,     2.45 tokens per second)
llama_perf_context_print:       total time =  173554.74 ms /   441 tokens
llama_perf_context_print:    graphs reused =        392


In [19]:
for i, query in enumerate(queries):
    responses.loc[len(responses)] = [
        "2.PROMPT_ENG_K_10",
        i+1,
        SYSTEM_PROMPT + "\n" + query,
        LLM_response(query=SYSTEM_PROMPT + "\n" + query, max_tokens= OPTIMAL_MAX_TOKENS, temperature=OPTIMAL_TEMPERATURE, top_k=10, top_p=OPTIMAL_TOP_P, print=True),
        "",
        ""
    ]

Llama.generate: 40 prefix-match hit, remaining 14 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =    2658.04 ms /    14 tokens (  189.86 ms per token,     5.27 tokens per second)
llama_perf_context_print:        eval time =  263397.16 ms /   654 runs   (  402.75 ms per token,     2.48 tokens per second)
llama_perf_context_print:       total time =  266710.09 ms /   668 tokens
llama_perf_context_print:    graphs reused =        632


Llama.generate: 40 prefix-match hit, remaining 32 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =    6053.14 ms /    32 tokens (  189.16 ms per token,     5.29 tokens per second)
llama_perf_context_print:        eval time =  115158.71 ms /   288 runs   (  399.86 ms per token,     2.50 tokens per second)
llama_perf_context_print:       total time =  121404.32 ms /   320 tokens
llama_perf_context_print:    graphs reused =        278


Llama.generate: 42 prefix-match hit, remaining 34 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =    6550.18 ms /    34 tokens (  192.65 ms per token,     5.19 tokens per second)
llama_perf_context_print:        eval time =  195653.19 ms /   494 runs   (  396.06 ms per token,     2.52 tokens per second)
llama_perf_context_print:       total time =  202620.55 ms /   528 tokens
llama_perf_context_print:    graphs reused =        478


Llama.generate: 40 prefix-match hit, remaining 28 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =    5275.36 ms /    28 tokens (  188.41 ms per token,     5.31 tokens per second)
llama_perf_context_print:        eval time =  135757.66 ms /   345 runs   (  393.50 ms per token,     2.54 tokens per second)
llama_perf_context_print:       total time =  141278.22 ms /   373 tokens
llama_perf_context_print:    graphs reused =        334


Llama.generate: 40 prefix-match hit, remaining 35 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =    6676.55 ms /    35 tokens (  190.76 ms per token,     5.24 tokens per second)
llama_perf_context_print:        eval time =  182388.05 ms /   452 runs   (  403.51 ms per token,     2.48 tokens per second)
llama_perf_context_print:       total time =  189439.13 ms /   487 tokens
llama_perf_context_print:    graphs reused =        437


## Data Preparation for RAG

### Data Load

In [20]:
#Load PDF 
pdf_loader = PyMuPDFLoader(PDF_PATH)
manual = pdf_loader.load()

### Data Overview

#### Checking the first 5 pages

In [21]:
for i in range(5):
    display(HTML(f"<h2>Page Number : {i+1}</h2><p>{manual[i].metadata}</p>"))
    display(HTML(f"Content:<pre style=\"white-space:pre-line;\">{manual[i].page_content.replace("\n","<br>")}</pre>"))

#### Checking the number of pages

In [22]:
display(HTML("<h2>Number of Pages</h2>"), len(manual))


4114

### Data Chunking

In [23]:
# Doing only 1 variation for Data Chunks = 1000, overlap =100, which is good enough. 
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
                    encoding_name=CHUNK_ENCODING,
                    chunk_size=CHUNK_SIZE,
                    chunk_overlap=CHUNK_OVERLAP 
                )

document_chunks = pdf_loader.load_and_split(text_splitter)
display("Number of Data chunks", len(document_chunks))

for i in range(4):
    display(HTML(f"<h2>Chunk {i+1}</h2>"),HTML(f"<pre style=\"white-space:pre-line;\">{ document_chunks[i].page_content.replace("\n","<br>") }</pre>"))

'Number of Data chunks'

4703

As expected, there are some overlaps

### Data Embedding

In [24]:
embedder = SentenceTransformerEmbeddings(model_name=EMBEDDING_MODEL) 

/tmp/ipykernel_23/3995730196.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedder = SentenceTransformerEmbeddings(model_name=EMBEDDING_MODEL)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [25]:
#Lets check the embedding 

embedding_1 = embedder.embed_query(document_chunks[0].page_content)
embedding_2 = embedder.embed_query(document_chunks[1].page_content)

# The Embedder model is all-MiniLM-L6-v2", so the dimensions must be 384 and embedding must match. 
display(HTML(f"<h3>Dimension of the embedding vector : {len(embedding_1)}"))
display(HTML(f"<h3>Embeddings match  : {len(embedding_1)==len(embedding_2)}"))

embedding_1,embedding_2

([-0.07390420138835907,
  0.10099316388368607,
  0.013632108457386494,
  -0.03955172374844551,
  0.0753292441368103,
  0.01078539714217186,
  0.010476995259523392,
  -0.007800430525094271,
  0.022440528497099876,
  0.009175014682114124,
  0.05784983932971954,
  0.019937070086598396,
  0.023529663681983948,
  -0.007724473252892494,
  -0.00505615770816803,
  0.005431563127785921,
  -0.07228931039571762,
  -0.041242122650146484,
  -0.04567202925682068,
  0.05360128730535507,
  0.020989274606108665,
  0.026765646412968636,
  -0.03706006705760956,
  0.0044648428447544575,
  -0.0004241139395162463,
  -0.0006763329147361219,
  -0.010279613547027111,
  0.011681376956403255,
  -0.040966346859931946,
  -0.0768781378865242,
  -0.0171975065022707,
  -0.0037196273915469646,
  -0.017689503729343414,
  0.057410191744565964,
  0.05917488783597946,
  -0.00447854632511735,
  0.0004533476894721389,
  0.01103211473673582,
  -0.028845978900790215,
  -0.01829986274242401,
  -0.013875748962163925,
  -0.05052

### Vector Database

In [26]:
# Create Vector Database with chunks. 

vector_db = Chroma.from_documents(
    document_chunks,
    embedder,
    persist_directory=VECTOR_DB
)

vector_db = Chroma(persist_directory=VECTOR_DB,embedding_function=embedder)

/tmp/ipykernel_23/3883667864.py:9: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(persist_directory=VECTOR_DB,embedding_function=embedder)


In [27]:
display(HTML("<h2>Embeddings<h2>"),vector_db.embeddings)

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

### Similarity Search Check

In [28]:
llm_test_query = "sepsis"
#Lets get top 3
documents = vector_db.similarity_search(llm_test_query, k=3) 
for i, document in enumerate(documents):
    display(HTML(f"<h2>Document {i+1}</h2>"))
    display(HTML(f"<h3>Document {i+1}: Metadata</h3>"), document.metadata)
    display(HTML(f"<h3>Document {i+1}: Content</h3>"),document.page_content)


{'author': '',
 'creationDate': 'D:20120615054440Z',
 'trapped': '',
 'keywords': '',
 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)',
 'subject': '',
 'page': 2453,
 'file_path': '/kaggle/input/datasets/assignarc/medical-dataset/medical_diagnosis_manual.pdf',
 'creationdate': '2012-06-15T05:44:40+00:00',
 'format': 'PDF 1.7',
 'moddate': '2026-03-04T00:19:39+00:00',
 'source': '/kaggle/input/datasets/assignarc/medical-dataset/medical_diagnosis_manual.pdf',
 'modDate': 'D:20260304001939Z',
 'creator': 'Atop CHM to PDF Converter',
 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition',
 'total_pages': 4114}

'Chapter 227. Sepsis and Septic Shock\nIntroduction\n(See also Ch. 226.)\nSepsis, severe sepsis, and septic shock are inflammatory states resulting from the systemic\nresponse to bacterial infection. In severe sepsis and septic shock, there is critical reduction in\ntissue perfusion. Common causes include gram-negative organisms, staphylococci, and\nmeningococci. Symptoms often begin with shaking chills and include fever, hypotension,\noliguria, and confusion. Acute failure of multiple organs, including the lungs, kidneys, and liver,\ncan occur. Treatment is aggressive fluid resuscitation, antibiotics, surgical excision of infected\nor necrotic tissues and drainage of pus, supportive care, and sometimes intensive control of\nblood glucose and administration of corticosteroids and activated protein C.\nA spectrum of severity exists (see\nTable 227-1).\nSepsis is infection accompanied by an acute inflammatory reaction with systemic manifestations\nassociated with release into the bloodst

{'modDate': 'D:20260304001939Z',
 'keywords': '',
 'total_pages': 4114,
 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition',
 'subject': '',
 'moddate': '2026-03-04T00:19:39+00:00',
 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)',
 'creator': 'Atop CHM to PDF Converter',
 'creationDate': 'D:20120615054440Z',
 'source': '/kaggle/input/datasets/assignarc/medical-dataset/medical_diagnosis_manual.pdf',
 'creationdate': '2012-06-15T05:44:40+00:00',
 'file_path': '/kaggle/input/datasets/assignarc/medical-dataset/medical_diagnosis_manual.pdf',
 'trapped': '',
 'format': 'PDF 1.7',
 'page': 1306,
 'author': ''}

'produces a thin, sterile fluid that is resorbed into the bloodstream. Incomplete resorption may leave a\ncystic loculation within a fibrous wall that may become calcified.\nIf the abscess is deep or if there is surrounding cellulitis, systemic antimicrobial drugs are indicated as\nadjunctive therapy; they are usually ineffective without drainage. Empiric antimicrobial therapy is based\non location and likely infecting pathogen. Gram stain, culture, and susceptibility results guide further\nantimicrobial therapy.\nBacteremia\n(See also Neonatal Sepsis on p. 2832 and Occult Bacteremia on p. 2841.)\nBacteremia is the presence of bacteria in the bloodstream. It can occur spontaneously, during\ncertain tissue infections, with use of indwelling GU or IV catheters, or after dental, GI, GU,\nwound-care, or other procedures. Bacteremia may cause metastatic infections, including\nendocarditis, especially in patients with valvular heart abnormalities. Transient bacteremia is\noften asymptomatic 

{'format': 'PDF 1.7',
 'file_path': '/kaggle/input/datasets/assignarc/medical-dataset/medical_diagnosis_manual.pdf',
 'creationdate': '2012-06-15T05:44:40+00:00',
 'modDate': 'D:20260304001939Z',
 'creationDate': 'D:20120615054440Z',
 'trapped': '',
 'author': '',
 'creator': 'Atop CHM to PDF Converter',
 'total_pages': 4114,
 'page': 1307,
 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)',
 'subject': '',
 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition',
 'source': '/kaggle/input/datasets/assignarc/medical-dataset/medical_diagnosis_manual.pdf',
 'moddate': '2026-03-04T00:19:39+00:00',
 'keywords': ''}

'shaking chills, persistent fever, altered sensorium, hypotension, and GI symptoms (abdominal pain,\nnausea, vomiting, diarrhea) suggests sepsis or septic shock. Septic shock develops in 25 to 40% of\npatients with significant bacteremia.\nDiagnosis\nIf bacteremia, sepsis, or septic shock is suspected, cultures are obtained of blood and any other\nappropriate specimens (see p. 1166).\nTreatment\n• Antibiotics\nIn patients with suspected bacteremia, empiric antibiotics are given after appropriate cultures are\nobtained. Early treatment of bacteremia with an appropriate antimicrobial regimen appears to improve\nsurvival. Continuing therapy involves adjusting antibiotics according to the results of culture and\nsusceptibility testing, surgically draining any abscesses, and usually removing any internal devices that\nare the suspected source of bacteria.\nBiological Warfare and Terrorism\nBiological warfare is the use of microbiological agents for hostile purposes. Such use is contrary to\

### Retriever Check

In [29]:
retriever = vector_db.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3} 
)
related_documents = retriever.invoke(llm_test_query) 
for i, document in enumerate(related_documents):
    display(HTML(f"<h2>Document {i+1}</h2>"))
    display(HTML(f"<h3>Document {i+1}: Metadata</h3>"), document.metadata)
    display(HTML(f"<h3>Document {i+1}: Content</h3>"),document.page_content)

{'creationDate': 'D:20120615054440Z',
 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition',
 'modDate': 'D:20260304001939Z',
 'moddate': '2026-03-04T00:19:39+00:00',
 'subject': '',
 'creationdate': '2012-06-15T05:44:40+00:00',
 'author': '',
 'creator': 'Atop CHM to PDF Converter',
 'format': 'PDF 1.7',
 'trapped': '',
 'total_pages': 4114,
 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)',
 'source': '/kaggle/input/datasets/assignarc/medical-dataset/medical_diagnosis_manual.pdf',
 'keywords': '',
 'page': 2453,
 'file_path': '/kaggle/input/datasets/assignarc/medical-dataset/medical_diagnosis_manual.pdf'}

'Chapter 227. Sepsis and Septic Shock\nIntroduction\n(See also Ch. 226.)\nSepsis, severe sepsis, and septic shock are inflammatory states resulting from the systemic\nresponse to bacterial infection. In severe sepsis and septic shock, there is critical reduction in\ntissue perfusion. Common causes include gram-negative organisms, staphylococci, and\nmeningococci. Symptoms often begin with shaking chills and include fever, hypotension,\noliguria, and confusion. Acute failure of multiple organs, including the lungs, kidneys, and liver,\ncan occur. Treatment is aggressive fluid resuscitation, antibiotics, surgical excision of infected\nor necrotic tissues and drainage of pus, supportive care, and sometimes intensive control of\nblood glucose and administration of corticosteroids and activated protein C.\nA spectrum of severity exists (see\nTable 227-1).\nSepsis is infection accompanied by an acute inflammatory reaction with systemic manifestations\nassociated with release into the bloodst

{'subject': '',
 'modDate': 'D:20260304001939Z',
 'source': '/kaggle/input/datasets/assignarc/medical-dataset/medical_diagnosis_manual.pdf',
 'creationdate': '2012-06-15T05:44:40+00:00',
 'trapped': '',
 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition',
 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)',
 'total_pages': 4114,
 'page': 1306,
 'moddate': '2026-03-04T00:19:39+00:00',
 'author': '',
 'creationDate': 'D:20120615054440Z',
 'creator': 'Atop CHM to PDF Converter',
 'keywords': '',
 'format': 'PDF 1.7',
 'file_path': '/kaggle/input/datasets/assignarc/medical-dataset/medical_diagnosis_manual.pdf'}

'produces a thin, sterile fluid that is resorbed into the bloodstream. Incomplete resorption may leave a\ncystic loculation within a fibrous wall that may become calcified.\nIf the abscess is deep or if there is surrounding cellulitis, systemic antimicrobial drugs are indicated as\nadjunctive therapy; they are usually ineffective without drainage. Empiric antimicrobial therapy is based\non location and likely infecting pathogen. Gram stain, culture, and susceptibility results guide further\nantimicrobial therapy.\nBacteremia\n(See also Neonatal Sepsis on p. 2832 and Occult Bacteremia on p. 2841.)\nBacteremia is the presence of bacteria in the bloodstream. It can occur spontaneously, during\ncertain tissue infections, with use of indwelling GU or IV catheters, or after dental, GI, GU,\nwound-care, or other procedures. Bacteremia may cause metastatic infections, including\nendocarditis, especially in patients with valvular heart abnormalities. Transient bacteremia is\noften asymptomatic 

{'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition',
 'creator': 'Atop CHM to PDF Converter',
 'moddate': '2026-03-04T00:19:39+00:00',
 'page': 1307,
 'modDate': 'D:20260304001939Z',
 'author': '',
 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)',
 'subject': '',
 'keywords': '',
 'creationdate': '2012-06-15T05:44:40+00:00',
 'format': 'PDF 1.7',
 'creationDate': 'D:20120615054440Z',
 'trapped': '',
 'source': '/kaggle/input/datasets/assignarc/medical-dataset/medical_diagnosis_manual.pdf',
 'total_pages': 4114,
 'file_path': '/kaggle/input/datasets/assignarc/medical-dataset/medical_diagnosis_manual.pdf'}

'shaking chills, persistent fever, altered sensorium, hypotension, and GI symptoms (abdominal pain,\nnausea, vomiting, diarrhea) suggests sepsis or septic shock. Septic shock develops in 25 to 40% of\npatients with significant bacteremia.\nDiagnosis\nIf bacteremia, sepsis, or septic shock is suspected, cultures are obtained of blood and any other\nappropriate specimens (see p. 1166).\nTreatment\n• Antibiotics\nIn patients with suspected bacteremia, empiric antibiotics are given after appropriate cultures are\nobtained. Early treatment of bacteremia with an appropriate antimicrobial regimen appears to improve\nsurvival. Continuing therapy involves adjusting antibiotics according to the results of culture and\nsusceptibility testing, surgically draining any abscesses, and usually removing any internal devices that\nare the suspected source of bacteria.\nBiological Warfare and Terrorism\nBiological warfare is the use of microbiological agents for hostile purposes. Such use is contrary to\

### LLM Response Check

In [30]:
model_output = LLM_response(query=llm_test_query,
                max_tokens=OPTIMAL_MAX_TOKENS, 
                temperature=OPTIMAL_TEMPERATURE, 
                top_p=OPTIMAL_TOP_P, 
                top_k=OPTIMAL_TOP_K, 
                print=True)

Llama.generate: 1 prefix-match hit, remaining 3 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =     683.64 ms /     3 tokens (  227.88 ms per token,     4.39 tokens per second)
llama_perf_context_print:        eval time =  418300.29 ms /  1023 runs   (  408.90 ms per token,     2.45 tokens per second)
llama_perf_context_print:       total time =  420392.35 ms /  1026 tokens
llama_perf_context_print:    graphs reused =        990


### RAG Response Function

In [31]:
def LLM_RAG_response(
        query:str,
        max_tokens:int=128,
        k: int=3,
        temperature:float=0.0,
        top_p:float=0.95,
        top_k:int=50,
        print :bool = True
    ):

    global QNA_SYSTEM_PROMPT,QNA_USER_MESSAGE_TEMPLATE
    
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.invoke(input=query,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)
    prompt = QNA_SYSTEM_PROMPT + '\n' + QNA_USER_MESSAGE_TEMPLATE.format(context=context_for_query, question=query) 
    
    try:
        model_rag_output = llm(
                            prompt=prompt,
                            max_tokens=max_tokens,
                            temperature=temperature,
                            top_p=top_p,
                            top_k=top_k
                        )
         # Extract and print the model's response
        rag_response = model_rag_output['choices'][0]['text'].strip()
        if(print):
            display(HTML(
                    f"<pre style=\"white-space:pre-line;\">Prompt : <br>{prompt.replace("\n","<br>")}</pre>" +
                    f"<h4>{llm.metadata['general.name']} | Tokens : {max_tokens} | Temp : {temperature} | top_p : {top_p}, | top_k : {top_k} </h4> " +
                    f"<pre style=\"white-space:pre-line;\">{rag_response.replace("\n","<p>")}</pre>"
                )
            )
    except Exception as e:
        rag_response = f'Sorry, I encountered the following error: \n {e}'

    return prompt, rag_response

## RAG without fine tuning

In [32]:
#Comment this when done with debugging, it keeps growing otherwise. 
#responses=  pd.DataFrame(columns=["Type","Run","Query","Response"])

for i, query in enumerate(queries):
    result = LLM_RAG_response(query=query,max_tokens= OPTIMAL_MAX_TOKENS, temperature=OPTIMAL_TEMPERATURE, top_k=OPTIMAL_TOP_K, top_p=OPTIMAL_TOP_P, print=True)
    responses.loc[len(responses)] = [
        "3.RAG_NO_TUNING",
        i+1,
        result[0],
        result[1],
        "",
        ""
    ]


Llama.generate: 1 prefix-match hit, remaining 3248 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  659934.99 ms /  3248 tokens (  203.18 ms per token,     4.92 tokens per second)
llama_perf_context_print:        eval time =  160162.27 ms /   341 runs   (  469.68 ms per token,     2.13 tokens per second)
llama_perf_context_print:       total time =  820344.63 ms /  3589 tokens
llama_perf_context_print:    graphs reused =        329


Llama.generate: 26 prefix-match hit, remaining 3206 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  643039.78 ms /  3206 tokens (  200.57 ms per token,     4.99 tokens per second)
llama_perf_context_print:        eval time =  294123.75 ms /   623 runs   (  472.11 ms per token,     2.12 tokens per second)
llama_perf_context_print:       total time =  937775.89 ms /  3829 tokens
llama_perf_context_print:    graphs reused =        603


Llama.generate: 26 prefix-match hit, remaining 2735 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  542991.15 ms /  2735 tokens (  198.53 ms per token,     5.04 tokens per second)
llama_perf_context_print:        eval time =  222779.81 ms /   476 runs   (  468.02 ms per token,     2.14 tokens per second)
llama_perf_context_print:       total time =  766182.17 ms /  3211 tokens
llama_perf_context_print:    graphs reused =        460


Llama.generate: 26 prefix-match hit, remaining 2819 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  569081.50 ms /  2819 tokens (  201.87 ms per token,     4.95 tokens per second)
llama_perf_context_print:        eval time =  121250.52 ms /   264 runs   (  459.28 ms per token,     2.18 tokens per second)
llama_perf_context_print:       total time =  690506.31 ms /  3083 tokens
llama_perf_context_print:    graphs reused =        254


Llama.generate: 26 prefix-match hit, remaining 2647 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  522405.93 ms /  2647 tokens (  197.36 ms per token,     5.07 tokens per second)
llama_perf_context_print:        eval time =  235241.18 ms /   512 runs   (  459.46 ms per token,     2.18 tokens per second)
llama_perf_context_print:       total time =  758099.87 ms /  3159 tokens
llama_perf_context_print:    graphs reused =        495


## RAG with fine-tuning

In [33]:
#Comment this when done with debugging, it keeps growing otherwise. 
#responses=  pd.DataFrame(columns=["Type","Run","Query","Response"])

for i, query in enumerate(queries):
    result = LLM_RAG_response(query=query,max_tokens= OPTIMAL_MAX_TOKENS, temperature=0.5, top_k=OPTIMAL_TOP_K, top_p=OPTIMAL_TOP_P, print=True)
    responses.loc[len(responses)] = [
        "4.RAG_TUNING_T_0.5",
        i+1,
        result[0],
        result[1],
        "",
        ""
    ]


Llama.generate: 26 prefix-match hit, remaining 3223 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  656697.38 ms /  3223 tokens (  203.75 ms per token,     4.91 tokens per second)
llama_perf_context_print:        eval time =  162438.48 ms /   341 runs   (  476.36 ms per token,     2.10 tokens per second)
llama_perf_context_print:       total time =  819391.50 ms /  3564 tokens
llama_perf_context_print:    graphs reused =        329


Llama.generate: 26 prefix-match hit, remaining 3206 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  656214.76 ms /  3206 tokens (  204.68 ms per token,     4.89 tokens per second)
llama_perf_context_print:        eval time =  295432.50 ms /   623 runs   (  474.21 ms per token,     2.11 tokens per second)
llama_perf_context_print:       total time =  952266.54 ms /  3829 tokens
llama_perf_context_print:    graphs reused =        603


Llama.generate: 26 prefix-match hit, remaining 2735 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  545124.78 ms /  2735 tokens (  199.31 ms per token,     5.02 tokens per second)
llama_perf_context_print:        eval time =  221537.46 ms /   476 runs   (  465.41 ms per token,     2.15 tokens per second)
llama_perf_context_print:       total time =  767077.27 ms /  3211 tokens
llama_perf_context_print:    graphs reused =        460


Llama.generate: 26 prefix-match hit, remaining 2819 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  552529.68 ms /  2819 tokens (  196.00 ms per token,     5.10 tokens per second)
llama_perf_context_print:        eval time =  120604.75 ms /   264 runs   (  456.84 ms per token,     2.19 tokens per second)
llama_perf_context_print:       total time =  673306.55 ms /  3083 tokens
llama_perf_context_print:    graphs reused =        254


Llama.generate: 26 prefix-match hit, remaining 2647 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  515575.95 ms /  2647 tokens (  194.78 ms per token,     5.13 tokens per second)
llama_perf_context_print:        eval time =  230977.98 ms /   512 runs   (  451.13 ms per token,     2.22 tokens per second)
llama_perf_context_print:       total time =  746999.57 ms /  3159 tokens
llama_perf_context_print:    graphs reused =        495


In [34]:
#Comment this when done with debugging, it keeps growing otherwise. 
#responses=  pd.DataFrame(columns=["Type","Run","Query","Response"])

for i, query in enumerate(queries):
    result = LLM_RAG_response(query=query,max_tokens= OPTIMAL_MAX_TOKENS, temperature=0.7, top_k=OPTIMAL_TOP_K, top_p=OPTIMAL_TOP_P, print=True)
    responses.loc[len(responses)] = [
        "4a.RAG_TUNING_T_0.7",
        i+1,
        result[0],
        result[1],
        "",
        ""
    ]


Llama.generate: 26 prefix-match hit, remaining 3223 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  630106.76 ms /  3223 tokens (  195.50 ms per token,     5.12 tokens per second)
llama_perf_context_print:        eval time =  148263.28 ms /   321 runs   (  461.88 ms per token,     2.17 tokens per second)
llama_perf_context_print:       total time =  778593.32 ms /  3544 tokens
llama_perf_context_print:    graphs reused =        310


Llama.generate: 26 prefix-match hit, remaining 3206 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  634512.04 ms /  3206 tokens (  197.91 ms per token,     5.05 tokens per second)
llama_perf_context_print:        eval time =  282530.23 ms /   623 runs   (  453.50 ms per token,     2.21 tokens per second)
llama_perf_context_print:       total time =  917645.54 ms /  3829 tokens
llama_perf_context_print:    graphs reused =        603


Llama.generate: 26 prefix-match hit, remaining 2735 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  539303.74 ms /  2735 tokens (  197.19 ms per token,     5.07 tokens per second)
llama_perf_context_print:        eval time =  214935.33 ms /   476 runs   (  451.54 ms per token,     2.21 tokens per second)
llama_perf_context_print:       total time =  754640.93 ms /  3211 tokens
llama_perf_context_print:    graphs reused =        460


Llama.generate: 26 prefix-match hit, remaining 2819 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  550566.94 ms /  2819 tokens (  195.31 ms per token,     5.12 tokens per second)
llama_perf_context_print:        eval time =  119543.49 ms /   264 runs   (  452.82 ms per token,     2.21 tokens per second)
llama_perf_context_print:       total time =  670283.08 ms /  3083 tokens
llama_perf_context_print:    graphs reused =        254


Llama.generate: 26 prefix-match hit, remaining 2647 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  532659.13 ms /  2647 tokens (  201.23 ms per token,     4.97 tokens per second)
llama_perf_context_print:        eval time =  241252.83 ms /   512 runs   (  471.20 ms per token,     2.12 tokens per second)
llama_perf_context_print:       total time =  774375.89 ms /  3159 tokens
llama_perf_context_print:    graphs reused =        495


In [35]:
#Comment this when done with debugging, it keeps growing otherwise. 
#responses=  pd.DataFrame(columns=["Type","Run","Query","Response"])

for i, query in enumerate(queries):
    result = LLM_RAG_response(query=query,max_tokens= OPTIMAL_MAX_TOKENS, temperature=OPTIMAL_TEMPERATURE, top_k=5, top_p=OPTIMAL_TOP_P, print=True)
    responses.loc[len(responses)] = [
        "4b.RAG_TUNING_K_5",
        i+1,
        result[0],
        result[1],
        "",
        ""
    ]


Llama.generate: 26 prefix-match hit, remaining 3223 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  659396.76 ms /  3223 tokens (  204.59 ms per token,     4.89 tokens per second)
llama_perf_context_print:        eval time =  164442.66 ms /   341 runs   (  482.24 ms per token,     2.07 tokens per second)
llama_perf_context_print:       total time =  824090.89 ms /  3564 tokens
llama_perf_context_print:    graphs reused =        329


Llama.generate: 26 prefix-match hit, remaining 3206 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  652284.60 ms /  3206 tokens (  203.46 ms per token,     4.92 tokens per second)
llama_perf_context_print:        eval time =  301882.96 ms /   623 runs   (  484.56 ms per token,     2.06 tokens per second)
llama_perf_context_print:       total time =  954798.26 ms /  3829 tokens
llama_perf_context_print:    graphs reused =        603


Llama.generate: 26 prefix-match hit, remaining 2735 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  556870.46 ms /  2735 tokens (  203.61 ms per token,     4.91 tokens per second)
llama_perf_context_print:        eval time =  224226.12 ms /   476 runs   (  471.06 ms per token,     2.12 tokens per second)
llama_perf_context_print:       total time =  781505.15 ms /  3211 tokens
llama_perf_context_print:    graphs reused =        460


Llama.generate: 26 prefix-match hit, remaining 2819 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  578004.33 ms /  2819 tokens (  205.04 ms per token,     4.88 tokens per second)
llama_perf_context_print:        eval time =  122883.62 ms /   264 runs   (  465.47 ms per token,     2.15 tokens per second)
llama_perf_context_print:       total time =  701062.21 ms /  3083 tokens
llama_perf_context_print:    graphs reused =        254


Llama.generate: 26 prefix-match hit, remaining 2647 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  520473.63 ms /  2647 tokens (  196.63 ms per token,     5.09 tokens per second)
llama_perf_context_print:        eval time =  233899.71 ms /   512 runs   (  456.84 ms per token,     2.19 tokens per second)
llama_perf_context_print:       total time =  754822.55 ms /  3159 tokens
llama_perf_context_print:    graphs reused =        495


In [36]:
#Comment this when done with debugging, it keeps growing otherwise. 
#responses=  pd.DataFrame(columns=["Type","Run","Query","Response"])

for i, query in enumerate(queries):
    result = LLM_RAG_response(query=query, max_tokens= OPTIMAL_MAX_TOKENS, temperature=OPTIMAL_TEMPERATURE, top_k=OPTIMAL_TOP_K, top_p=0.8, print=True)
    responses.loc[len(responses)] = [
        "4c.RAG_TUNING_P_0.8",
        i+1,
        result[0],
        result[1],
        "",
        ""
    ]

Llama.generate: 26 prefix-match hit, remaining 3223 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  639769.17 ms /  3223 tokens (  198.50 ms per token,     5.04 tokens per second)
llama_perf_context_print:        eval time =  196430.22 ms /   415 runs   (  473.33 ms per token,     2.11 tokens per second)
llama_perf_context_print:       total time =  836534.99 ms /  3638 tokens
llama_perf_context_print:    graphs reused =        401


Llama.generate: 26 prefix-match hit, remaining 3206 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  654857.27 ms /  3206 tokens (  204.26 ms per token,     4.90 tokens per second)
llama_perf_context_print:        eval time =  192962.70 ms /   409 runs   (  471.79 ms per token,     2.12 tokens per second)
llama_perf_context_print:       total time =  848144.09 ms /  3615 tokens
llama_perf_context_print:    graphs reused =        396


Llama.generate: 26 prefix-match hit, remaining 2735 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  554406.08 ms /  2735 tokens (  202.71 ms per token,     4.93 tokens per second)
llama_perf_context_print:        eval time =  157089.11 ms /   333 runs   (  471.74 ms per token,     2.12 tokens per second)
llama_perf_context_print:       total time =  711739.09 ms /  3068 tokens
llama_perf_context_print:    graphs reused =        322


Llama.generate: 26 prefix-match hit, remaining 2819 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  559675.66 ms /  2819 tokens (  198.54 ms per token,     5.04 tokens per second)
llama_perf_context_print:        eval time =  125264.52 ms /   271 runs   (  462.23 ms per token,     2.16 tokens per second)
llama_perf_context_print:       total time =  685120.68 ms /  3090 tokens
llama_perf_context_print:    graphs reused =        261


Llama.generate: 26 prefix-match hit, remaining 2647 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  516832.43 ms /  2647 tokens (  195.25 ms per token,     5.12 tokens per second)
llama_perf_context_print:        eval time =  255204.20 ms /   544 runs   (  469.13 ms per token,     2.13 tokens per second)
llama_perf_context_print:       total time =  772545.91 ms /  3191 tokens
llama_perf_context_print:    graphs reused =        526


## LLM-as-a-judge - Output Evaluation

### Grounding Function

In [37]:
def LLM_grounding_relevance_response(query:str,
        k=3,
        max_tokens=128,
        temperature=0,
        top_p=0.95,
        top_k=50,
        print :bool = True):
    global QNA_SYSTEM_PROMPT,QNA_USER_MESSAGE_TEMPLATE, GROUNDNESS_USER_MESSAGE_TEMPLATE
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.invoke(input=query,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{QNA_SYSTEM_PROMPT}\n
                {'user'}: {QNA_USER_MESSAGE_TEMPLATE.format(context=context_for_query, question=query)}
                [/INST]"""

    llm_response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            echo=False
            )

    answer =  llm_response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{GROUNDNESS_RATE_SYSTEM_MESSAGE}\n
                            {'user'}: {GROUNDNESS_USER_MESSAGE_TEMPLATE.format(context=context_for_query, question=query, answer=answer)}
                            [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{RELEVENCE_RATER_SYSTEM_MESSAGE}\n
                        {'user'}: {GROUNDNESS_USER_MESSAGE_TEMPLATE.format(context=context_for_query, question=query, answer=answer)}
                        [/INST]"""

    grounding_response = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            echo=False
            )

    relevence_response = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            echo=False
            )

    if(print):
        display(HTML(
                f"<pre style=\"white-space:pre-line;\">Query: <br>{query}</pre>" +
                f"<h4>{llm.metadata['general.name']} </h4> " +
                f"<pre style=\"white-space:pre-line;\">{answer.replace("\n","<br>")}</pre>" +
                f"<pre style=\"white-space:pre-line;\">Groundedness Prompt: {groundedness_prompt.replace("\n","<br>")}</pre>" +
                f"<pre style=\"white-space:pre-line;\">Relevance Prompt: {relevance_prompt.replace("\n","<br>")}</pre>" +
                f"<pre style=\"white-space:pre-line;\">Groundedness: {grounding_response['choices'][0]['text'].replace("\n","<br>")}</pre>" +
                f"<pre style=\"white-space:pre-line;\">Relevance: {relevence_response['choices'][0]['text'].replace("\n","<br>")}</pre>"

            )
        )

    return prompt,answer,grounding_response['choices'][0]['text'],relevence_response['choices'][0]['text']


### LLM-as-a-judge

In [38]:
#Comment this when done with debugging, it keeps growing otherwise. 
#responses=  pd.DataFrame(columns=["Type","Run","Query","Response", "Grounding","Relevance"])

for i, query in enumerate(queries):
    result = LLM_grounding_relevance_response(query=query,max_tokens= OPTIMAL_MAX_TOKENS, temperature=OPTIMAL_TEMPERATURE, top_k=OPTIMAL_TOP_K, top_p=OPTIMAL_TOP_P, print=True)
    responses.loc[len(responses)] = [
        "5.GROUND_RELEVANCE",
        i+1,
        result[0],
        result[1], 
        result[2],
        result[3]
    ]


Llama.generate: 1 prefix-match hit, remaining 3263 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  661933.03 ms /  3263 tokens (  202.86 ms per token,     4.93 tokens per second)
llama_perf_context_print:        eval time =  178058.99 ms /   371 runs   (  479.94 ms per token,     2.08 tokens per second)
llama_perf_context_print:       total time =  840278.39 ms /  3634 tokens
llama_perf_context_print:    graphs reused =        359
Llama.generate: 4 prefix-match hit, remaining 3874 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  791815.70 ms /  3874 tokens (  204.39 ms per token,     4.89 tokens per second)
llama_perf_context_print:        eval time =   76964.26 ms /   157 runs   (  490.22 ms per token,     2.04 tokens per second)
llama_perf_context_print:       total time =  868868.93 ms /  4031 tokens
llama_perf_context_print:   

Llama.generate: 4 prefix-match hit, remaining 3243 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  662425.63 ms /  3243 tokens (  204.26 ms per token,     4.90 tokens per second)
llama_perf_context_print:        eval time =  200729.55 ms /   435 runs   (  461.45 ms per token,     2.17 tokens per second)
llama_perf_context_print:       total time =  863509.28 ms /  3678 tokens
llama_perf_context_print:    graphs reused =        420
Llama.generate: 4 prefix-match hit, remaining 3922 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  804525.70 ms /  3922 tokens (  205.13 ms per token,     4.87 tokens per second)
llama_perf_context_print:        eval time =  367570.63 ms /   746 runs   (  492.72 ms per token,     2.03 tokens per second)
llama_perf_context_print:       total time = 1172923.35 ms /  4668 tokens
llama_perf_context_print:   

Llama.generate: 4 prefix-match hit, remaining 2772 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  545422.25 ms /  2772 tokens (  196.76 ms per token,     5.08 tokens per second)
llama_perf_context_print:        eval time =  229815.69 ms /   501 runs   (  458.71 ms per token,     2.18 tokens per second)
llama_perf_context_print:       total time =  775680.97 ms /  3273 tokens
llama_perf_context_print:    graphs reused =        484
Llama.generate: 4 prefix-match hit, remaining 3517 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  718520.73 ms /  3517 tokens (  204.30 ms per token,     4.89 tokens per second)
llama_perf_context_print:        eval time =   92425.70 ms /   193 runs   (  478.89 ms per token,     2.09 tokens per second)
llama_perf_context_print:       total time =  811061.65 ms /  3710 tokens
llama_perf_context_print:   

Llama.generate: 4 prefix-match hit, remaining 2856 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  584599.77 ms /  2856 tokens (  204.69 ms per token,     4.89 tokens per second)
llama_perf_context_print:        eval time =  309212.62 ms /   663 runs   (  466.38 ms per token,     2.14 tokens per second)
llama_perf_context_print:       total time =  894515.51 ms /  3519 tokens
llama_perf_context_print:    graphs reused =        641
Llama.generate: 4 prefix-match hit, remaining 3763 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  784000.12 ms /  3763 tokens (  208.34 ms per token,     4.80 tokens per second)
llama_perf_context_print:        eval time =  179029.15 ms /   370 runs   (  483.86 ms per token,     2.07 tokens per second)
llama_perf_context_print:       total time =  963313.76 ms /  4133 tokens
llama_perf_context_print:   

Llama.generate: 4 prefix-match hit, remaining 2684 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  533536.37 ms /  2684 tokens (  198.78 ms per token,     5.03 tokens per second)
llama_perf_context_print:        eval time =  205340.15 ms /   446 runs   (  460.40 ms per token,     2.17 tokens per second)
llama_perf_context_print:       total time =  739258.58 ms /  3130 tokens
llama_perf_context_print:    graphs reused =        432
Llama.generate: 4 prefix-match hit, remaining 3374 prompt tokens to eval
llama_perf_context_print:        load time =    4455.62 ms
llama_perf_context_print: prompt eval time =  662467.91 ms /  3374 tokens (  196.34 ms per token,     5.09 tokens per second)
llama_perf_context_print:        eval time =   66202.36 ms /   141 runs   (  469.52 ms per token,     2.13 tokens per second)
llama_perf_context_print:       total time =  728747.89 ms /  3515 tokens
llama_perf_context_print:   

## All Outputs

In [39]:
summary = responses.copy().sort_values(by=['Run','Type'])

for index, query in enumerate(queries):
    display(HTML(f"<h2>Query: {query}</h2>"))
    temp = summary[summary['Run'] == index+1]
    for _, response in temp.iterrows():
        display(HTML(
            f"<h3>{response['Type']} - Responses</h3>" + 
            f"<pre style=\"white-space:pre-line;\">Answer : <br> {response['Query'].replace("\n","<br>")}</pre>" + 
            f"<pre style=\"white-space:pre-line;\">Response: <br>{response['Response'].replace("\n","<br>")}</pre>" +
            f"<pre style=\"white-space:pre-line;\">Grounding: <br>{response['Grounding'].replace("\n","<br>")}</pre>" +
            f"<pre style=\"white-space:pre-line;\">Relevance: <br>{response['Relevance'].replace("\n","<br>")}</pre>" 
        ))

   ## Trucnated full response due to Kaggle 1MB upload limit, but is it just repeat print from above. 

## Actionable Insights and Business Recommendations

### Overview of Tuning Combinations
To optimize accuracy, constraints, and professional tone, the underlying AI models and RAG pipeline were tested across the following 10 parameter combinations:

| Run Key | Description |
| :--- | :--- |
| `1.TEST` | Baseline foundational LLM query with no grounding and default parameters. |
| `2.PROMPT_ENGINEERING` | Baseline prompt-engineered LLM constraint ensuring medical professionalism. |
| `2.PROMPT_ENG_T_0.7` | Prompt engineering with High Temperature (0.7) for increased response variance. |
| `2.PROMPT_ENG_P_0.8` | Prompt engineering with Nucleus Sampling (Top P = 0.8) for diverse but coherent text. |
| `2.PROMPT_ENG_K_10` | Prompt engineering with constrained Top K (10) for strict token probability selection. |
| `3.RAG_NO_TUNING` | RAG implemented with standard dense embeddings retrieval and baseline LLM generation. |
| `4.RAG_TUNING_T_0.5` | RAG with optimized Temperature (0.5) balancing factual recall and smooth articulation. |
| `4a.RAG_TUNING_T_0.7` | RAG with High Temperature (0.7) testing creative bounds against structured context. |
| `4b.RAG_TUNING_K_5` | RAG with strict semantic search (Top K = 5 chunks) limiting context window noise. |
| `4c.RAG_TUNING_P_0.8` | RAG with Nucleus Sampling (Top P = 0.8) adjusting probability mass of retrieved integration. |
| `5.GROUND_RELEVANCE` | Automated LLM-as-a-judge diagnostic evaluating the RAG responses for groundedness and relevance. |

---


### Key Takeaways for the Business



**1. Overcoming Information Overload & Streamlining Diagnostics:**
The transition from a raw Large Language Model (`TEST`, `PROMPT_ENGINEERING` variants) to a Retrieval-Augmented Generation system (`RAG` variants) demonstrates a profound ability to cut through noise. By indexing the core medical corpus and retrieving only the most pertinent document chunks (especially observed in strict configurations like `RAG_TUNING_K_5`), the prototype distills extensive medical texts into immediate, context-aware answers. This directly streamlines the diagnostic process, equipping healthcare professionals with rapid insights without manual research, preserving critical time in emergency settings.

**2. Impact on Diagnostics and Patient Outcomes:**
The evaluations (`GROUND_RELEVANCE` run) show consistently high relevance and groundedness in the RAG-generated responses. Unlike ungrounded base models which may hallucinate, the RAG system successfully identified symptoms for appendicitis, sudden patchy hair loss, brain injuries, and fractures based thoroughly on the retrieved corpus. Narrowing the semantic retrieval scope (`RAG_TUNING_K_5`) and tuning generation parameters (`RAG_TUNING_T_0.5`) drastically reduced AI hallucination risks. This accuracy profoundly impacts patient outcomes by supporting evidence-based, reliable clinical decision-making.

**3. Standardizing Care Practices:**
Using formal Prompt Engineering integrated directly into the RAG system ensures responses are delivered uniformly. By anchoring answers to the same centralized, gold-standard knowledge repository, healthcare institutions democratize access to high-standard medical protocols across departments. The comparison across the 5 LLM prompt tuning combinations proved that while generation styles can be tweaked (like `PROMPT_ENG_T_0.7`), enforcing a strict system prompt standardizes care practices and guarantees a concise, professional tone across the organization.

**4. Prototype Feasibility and Effectiveness:**
The RAG pipeline effectively chains dynamic chunking strategies, dense embeddings, vector search (`Chroma`), and localized LLM inference. The rigorous testing across 10 independent combinations proves both the technical capabilities and operational feasibility of this solution. The independent evaluation modules for "Groundedness" and "Relevance" provide a built-in auditing mechanism, ensuring the system remains continuously transparent, effective, and trustworthy for deployment in high-stakes clinical environments. 

**Conclusion:**
Implementing this tuned RAG-based AI solution stands to transform healthcare data accessibility. Not only does it mitigate information overload and accelerate time-to-diagnosis, but it also creates a verifiable, standardized bedrock of medical knowledge, addressing all primary business objectives and setting a new paradigm for AI-assisted patient care.

**1. Streamlined Decision-Making via Contextual Grounding**
By applying a Retrieval-Augmented Generation (RAG) framework, the AI solution successfully mitigates information overload. Clinicians no longer need to manually sift through the 4000+ page Merck Manuals; the system instantly retrieves highly relevant chunks and synthesizes them into actionable medical guidance (e.g., protocols for sepsis or appendicitis surgical procedures).

**2. RAG vs Non-RAG Output Quality Comparison**
A direct comparison between the queries demonstrates the value of RAG. When evaluating queries without RAG, the LLM relied solely on its pre-trained general knowledge, resulting in generic or summarized advice that often failed to include specific clinical protocols. When the exact same queries were processed using RAG, the model answered by explicitly extracting the clinical guidelines from the Merck Manuals. For instance, the RAG output accurately mapped out specific diagnostic signs (e.g., epigastric pain shifting to the right lower quadrant) and explicit emergency steps, entirely bypassing the risk of the model hallucinating medical data.

**3. Impact on Diagnostics and Patient Outcomes**
Because the RAG system provides precise, document-grounded protocols, this level of diagnostic accuracy directly impacts patient outcomes. It reduces diagnostic errors and accelerates time-to-treatment in critical care environments by supplying clinicians with immediate, validated reference material.

**4. Standardizing Care Practices**
The evaluations (using the LLM-as-a-judge method) confirmed high relevance and groundedness scores for the RAG responses. This demonstrates the prototype's potential to standardize care practices across diverse medical facilities. By anchoring responses strictly to an authorized medical corpus, the system ensures that all practitioners, regardless of experience level, have immediate access to Gold Standard medical protocols.

**5. Feasibility and Effectiveness of the Prototype**
The functional prototype demonstrates high feasibility for real-world deployment. The integration of PyMuPDFLoader, RecursiveCharacterTextSplitter, and Chroma allowed for efficient ingestion, chunking, and semantic search of complex medical texts. Fine-tuning the prompt engineering further refined the assistant’s tone, proving that low-code AI solutions can be highly effective in specialized domains.

**6. Future Integration and Scalability**
To maximize business impact, this RAG prototype should be integrated into existing Electronic Health Record (EHR) systems or telehealth platforms. Future iterations should focus on expanding the vector database to include continuously updated medical journals and pharmacological databases, ensuring the AI remains a state-of-the-art decision-support tool.
**7. Fine-Tuning/Prompt Engineering Impact**
When evaluating responses with and without fine-tuning (prompt engineering vs zero-shot), we observe that fine-tuning the system prompt dictates the persona and strictness of the LLM. Without prompt engineering, the model provides general advisory text. By adding a system prompt like 'You are a professional medical evaluator...', the LLM strictly conforms to the requested format and medical tone. This fine-tuning through prompt engineering ensures the generation remains concise, directly relevant to the query, and formatted effectively for immediate professional review.



## Export

In [40]:
# PDF conversion often fails due to missing TeX (LaTeX) dependencies. 
# Using 'webpdf' is a more reliable alternative that uses a headless browser.
!jupyter nbconvert --to html  "NLP_RAG_Project_Notebook.ipynb"

/usr/local/lib/python3.12/dist-packages/mistune.py:435: SyntaxWarning: invalid escape sequence '\|'
  cells[i][c] = re.sub('\\\\\|', '|', cell)
/usr/local/lib/python3.12/dist-packages/nbconvert/filters/filter_links.py:36: SyntaxWarning: invalid escape sequence '\_'
  text = re.sub(r'_', '\_', text) # Escape underscores in display text
[NbConvertApp] WARNING | pattern 'NLP_RAG_Project_Notebook.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-